# QUELLE RÉPARTITION DES PRÊTS À LA MÉDIATHÈQUE ?

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime as dt
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import kiblib.utils.lucas_sns_params
from kiblib.utils.db import DbConn

In [2]:
maj = dt.today().strftime('%d-%m-%Y à %Hh%M' )
print(f"Dernière mise à jour le {maj}")

Dernière mise à jour le 23-10-2026 à 12h39


In [3]:
db_conn = DbConn().create_engine()

In [4]:
query = """SELECT *
FROM koha_prod.items"""

In [5]:
items = pd.read_sql(query,db_conn)

In [6]:
cols2print = ["notforloan","damaged","damaged_on","barcode","location","ccode"]

In [7]:
# EXEMPLAIRES
items["Exemplaires"] = 1

In [8]:
# ACQUISITIONS
items.loc[items["dateaccessioned"].astype("Datetime64").between("2026-01-01","2026-12-31"),"Acquisitions"] = 1

In [20]:
# DOCUMENTS A DISPOSITION DU PUBLIC
pret_direct = ["MED0C","MED1A","MED2A","MED3A"]
pret_indirect = ["MED0A","MED2C","MED2D","MED3C","MED3D","MED3E","MED3F","MED3G","MED3H","MED3I"]
consultation = ["MED0B","MED2B"]

# Empruntables
items.loc[items["notforloan"]==0,"Exemplaires empruntables"] = 1

# En accès direct
items.loc[~(items["location"].isin(pret_indirect))&
          ~(items["location"].isin(consultation)),
          "Exemplaires en accès direct"] = 1

# Consultation uniquement
items.loc[items["location"].isin(consultation),
         "Exemplaires en consultation sur place uniquement"] = 1

# En accès indirect
items.loc[items["location"].isin(pret_indirect),
         "Exemplaires en accès indirect"] = 1

In [10]:
# DOCUMENTS DANS LE CIRCUIT INTERNE

items.loc[items["notforloan"]==-1,"Exemplaires en commande"] = 1
items.loc[items["notforloan"]==-2,"Exemplaires en traitement"] = 1
items.loc[items["damaged"]!=0,"Exemplaires abîmés"] = 1
items.loc[items["notforloan"]==-4,"Exemplaires en réparation"] = 1
items.loc[items["notforloan"]==5,"Exemplaires en reliure"] = 1
items.loc[items["itemlost"]==2,"Exemplaires perdus"] = 1
items.loc[items["itemlost"]==1,"Exemplaires non-restitués"] = 1

In [11]:
# DOCUMENTS ÉLIMINÉS

items.loc[(items["notforloan"]==4) &
          (items["itemlost"]==1),"Exemplaires éliminés non-restitués"] = 1

items.loc[(items["notforloan"]==4)&
          (items["itemlost"]==2),
          "Exemplaires éliminés perdus"] = 1


items.loc[(items["notforloan"]==4)&
          (items["withdrawn"]==1),
          "Exemplaires éliminés desherbés"] = 1


items.loc[(items["notforloan"]==4)&
          (items["damaged"]==1),
          "Exemplaires éliminés abîmés"] = 1

In [12]:
site_med = ["MED0B",
            "MED0C",
            "MED1A",
            "MED2A",
            "MED2B",
            "MED2C",
            "MED2D",
            "MED3A",
            "MED3B",
            "MED3C",
            "MED3D",
            "MED3E",
            "MED3F",
            "MED3G",
            "MED3H",
            "MED3I",
            "MED3J",
            "MED3K",
            "MED3L",
            "MED3M",
            "MED3N",
            "MED3O",
            "MED3P",
            "MED3Q",
            "MED3R",
            "MED3S",
            "MED3T"]

site_bus = ["BUS1A"]

site_col = ["MED0A"]

In [13]:
# SITES

# Médiathèque
items.loc[(items["homebranch"]=="MED")&
          (items["location"].isin(site_med)),"Site"] = "Médiatèque"

# Zébron
items.loc[items["homebranch"]=="BUS","Site"] = "Zébron"

# Collectivités
items.loc[items["location"]=="MED0A","Site"] = "Collectivités"

In [52]:
cols2keep = ["Site",
             "ccode",
             "location",
             'Exemplaires', 'Acquisitions',
             'Exemplaires empruntables', 'Exemplaires en accès direct',
             'Exemplaires en consultation sur place uniquement',
             'Exemplaires en accès indirect', 'Exemplaires en commande',
             'Exemplaires en traitement', 'Exemplaires abîmés',
             'Exemplaires en réparation', 'Exemplaires en reliure',
             'Exemplaires perdus', 'Exemplaires non-restitués',
             'Exemplaires éliminés non-restitués', 'Exemplaires éliminés perdus',
             'Exemplaires éliminés desherbés', 'Exemplaires éliminés abîmés']

indicateurs = ['Exemplaires', 'Acquisitions',
              'Exemplaires empruntables', 'Exemplaires en accès direct',
              'Exemplaires en consultation sur place uniquement',
              'Exemplaires en accès indirect', 'Exemplaires en commande',
              'Exemplaires en traitement', 'Exemplaires abîmés',
              'Exemplaires en réparation', 'Exemplaires en reliure',
              'Exemplaires perdus', 'Exemplaires non-restitués',
              'Exemplaires éliminés non-restitués', 'Exemplaires éliminés perdus',
              'Exemplaires éliminés desherbés', 'Exemplaires éliminés abîmés']

In [53]:
for indicateur in indicateurs:
    print(indicateur)
    
    df = analyse[]

Exemplaires
Acquisitions
Exemplaires empruntables
Exemplaires en accès direct
Exemplaires en consultation sur place uniquement
Exemplaires en accès indirect
Exemplaires en commande
Exemplaires en traitement
Exemplaires abîmés
Exemplaires en réparation
Exemplaires en reliure
Exemplaires perdus
Exemplaires non-restitués
Exemplaires éliminés non-restitués
Exemplaires éliminés perdus
Exemplaires éliminés desherbés
Exemplaires éliminés abîmés


In [59]:
analyse = items[cols2keep]

In [76]:
analyse_med = analyse[(analyse["Site"]=="Médiatèque")
                     &(analyse["location"].isin(pret_direct))
                     ]

In [77]:
analyse_med = analyse_med.pivot_table(index="location",values=indicateurs,aggfunc=sum)

In [78]:
analyse_med

,Acquisitions,Exemplaires,Exemplaires abîmés,Exemplaires empruntables,Exemplaires en accès direct,Exemplaires en accès indirect,Exemplaires en commande,Exemplaires en consultation sur place uniquement,Exemplaires en reliure,Exemplaires en réparation,Exemplaires en traitement,Exemplaires non-restitués,Exemplaires perdus,Exemplaires éliminés abîmés,Exemplaires éliminés desherbés,Exemplaires éliminés non-restitués,Exemplaires éliminés perdus
location,,,,,,,,,,,,,,,,,
MED0C,611.0,10951,19.0,10372.0,10951.0,0.0,0.0,0.0,0.0,5.0,7.0,371.0,16.0,11.0,16.0,333.0,16.0
MED1A,3724.0,67237,575.0,63778.0,67237.0,0.0,178.0,0.0,0.0,73.0,113.0,1159.0,136.0,483.0,909.0,1014.0,123.0
MED2A,3035.0,40199,1427.0,34145.0,40199.0,0.0,0.0,0.0,0.0,235.0,264.0,4030.0,61.0,1112.0,273.0,3719.0,44.0
MED3A,778.0,15455,39.0,14370.0,15455.0,0.0,0.0,0.0,0.0,20.0,14.0,741.0,15.0,15.0,198.0,707.0,10.0
